In [68]:
import torch
import joblib
import pickle
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import vit_b_16
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

#### Loading datasets

In [14]:
data = joblib.load("datasets2.pkl")

X_test_flat = data["X_test_flat"]
X_test_rad = data["X_test_rad"]
y_test = data["y_test"].astype(int)

In [18]:
X_test_img = X_test_flat.reshape(-1,1,224,224)

#### Random Forest

In [33]:
with open("preprocess3.pkl", "rb") as f:
    prep = pickle.load(f)

scaler_flat = prep["scaler_flat"]
scaler_rad  = prep["scaler"]
pca         = prep["pca"]

In [34]:
pca = prep["pca"]
print(pca.n_components_)

135


In [35]:
X_test_flat_scaled = scaler_flat.transform(X_test_flat)

X_test_pca = pca.transform(X_test_flat_scaled)

y_proba_pca = rf_pca.predict_proba(X_test_pca)[:, 1]

In [39]:
X_test_mix = np.hstack([X_test_flat, X_test_rad])

In [16]:
rf_pca = joblib.load("rf_pca2.pkl")
rf_rad = joblib.load("rf_rad2.pkl")
rf_mix = joblib.load("rf_mix2.pkl")

#### Ewaluacja

In [25]:
def evaluate_model(y_true, proba, threshold=0.4):
    y_pred = (proba >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "y_pred": y_pred,
        "proba": proba
    }


In [59]:
X_test_flat_scaled = scaler_flat.transform(X_test_flat)
X_test_rad = scaler_rad.transform(X_test_rad)
X_test_pca = pca.transform(X_test_flat_scaled)

y_proba_pca = rf_pca.predict_proba(X_test_pca)[:, 1]
y_proba_rad = rf_rad.predict_proba(X_test_rad)[:, 1]

In [66]:
results = {}

results["RF_PCA"] = evaluate_model(
    y_test,
    rf_pca.predict_proba(X_test_pca)[:,1]
)

results["RF_Radiomics"] = evaluate_model(
    y_test,
    rf_rad.predict_proba(X_test_rad)[:,1]
)

#### Ewaluacja CNN 

In [43]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32*(224//4)*(224//4),64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,2)
        )

    def forward(self,x):
        return self.net(x)

cnn = CNN().to(device)
cnn.load_state_dict(torch.load("cnn_model2.pt", map_location=device))
cnn.eval()


C:\Users\Dell\AppData\Local\Temp\ipykernel_3096\2653241371.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("cnn_model2.pt", map_location=

CNN(
  (net): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=100352, out_features=64, bias=True)
    (8): ReLU()
    (9): Dropout(p=0.3, inplace=False)
    (10): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [45]:
X_test_img = torch.tensor(X_test_flat.reshape(-1,1,224,224), dtype=torch.float32)

with torch.no_grad():
    logits = cnn(X_test_img.to(device))
    proba_cnn = torch.softmax(logits, dim=1)[:,1].cpu().numpy()


In [61]:
results["CNN"] = evaluate_model(y_test, proba_cnn, threshold=0.4)

#### ViT ewaluacja

In [46]:
from torchvision.models import vit_b_16

vit = vit_b_16(weights=None)
vit.heads.head = nn.Linear(vit.heads.head.in_features, 2)

vit.load_state_dict(torch.load("vit_model.pt", map_location=device))
vit = vit.to(device)
vit.eval()

X_test_vit = X_test_img.repeat(1,3,1,1)  # 1 → 3 kanały


C:\Users\Dell\AppData\Local\Temp\ipykernel_3096\3298691711.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vit.load_state_dict(torch.load("vit_model.pt", map_location=de

In [48]:
with torch.no_grad():
    logits = vit(X_test_vit.to(device))
    proba_vit = torch.softmax(logits, dim=1)[:,1].cpu().numpy()


In [62]:
results["ViT"] = evaluate_model(y_test, proba_vit, threshold=0.4)

#### Final

In [63]:
results

{'RF_PCA': {'Accuracy': 0.40476190476190477,
  'Precision': 0.38387096774193546,
  'Recall': 0.9296875,
  'F1': 0.54337899543379,
  'ROC-AUC': 0.5588191105769231,
  'y_pred': array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
         1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
         1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
         0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
  

In [64]:
df_results = pd.DataFrame(results).T.drop(columns=["y_pred","proba"])
df_results

,Accuracy,Precision,Recall,F1,ROC-AUC
RF_PCA,0.404762,0.383871,0.929688,0.543379,0.558819
RF_Radiomics,0.619048,0.0,0.0,0.0,0.546856
CNN,0.583333,0.38,0.148438,0.213483,0.542368
ViT,0.633929,0.567568,0.164062,0.254545,0.617225
